# Return Probability Classifier — XGBoost

**Depends on:** `data/master_df_v2.parquet` — produced by `02_clustering.ipynb`  

**Outputs:**
- `data/master_df_v3.parquet` — master_df_v2 + `return_prob` + `return_flag`
- `outputs/return_clf_v1.pkl` — fitted model
- `outputs/return_classifier_metrics.json` — ROC-AUC, PR-AUC, Brier, threshold metrics
- `outputs/roc_pr_curves.png`
- `outputs/feature_importance.png`
- `outputs/return_prob_distribution.png`

---

All heavy logic lives in `src/return_classifier.py`. This notebook calls it, then focuses on evaluation visualisation.


In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    RocCurveDisplay, PrecisionRecallDisplay,
    ConfusionMatrixDisplay, confusion_matrix,
)

from src.return_classifier import (
    build_features,
    train,
    evaluate,
    predict_proba,
    add_return_prob,
    save_model,
    save_metrics,
    CAT_COLS, NUM_COLS, TARGET, RETURN_PROB_THRESHOLD,
)

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

DATA_DIR   = '../data'
OUTPUT_DIR = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Imports OK.')


---
## 1. Load data


In [ ]:
df = pd.read_parquet(f'{DATA_DIR}/master_df_v2.parquet')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')


---
## 2. Class imbalance check

Target is `is_return`. Expect ~3-5% positive rate.


In [ ]:
pos_rate = df[TARGET].mean()
n_pos    = df[TARGET].sum()
n_neg    = len(df) - n_pos

print(f'Total rows    : {len(df):,}')
print(f'Positives (1) : {n_pos:,}  ({pos_rate*100:.2f}%)')
print(f'Negatives (0) : {n_neg:,}  ({(1-pos_rate)*100:.2f}%)')
print(f'Implied scale_pos_weight : {n_neg/n_pos:.1f}')
print()
print('Note: heavy imbalance — ROC-AUC alone is misleading.')
print('      PR-AUC is the primary evaluation metric.')

---
## 3. Feature matrix


In [ ]:
X = build_features(df)
y = df[TARGET].astype(int)

print(f'X shape : {X.shape}')
print(f'Dtypes  :\n{X.dtypes}')
print()
X.describe().round(3)

---
## 4. Train

XGBClassifier with `scale_pos_weight` auto-computed from training split.  
Wrapped in `CalibratedClassifierCV(method='sigmoid', cv=3)` for probability calibration.


In [ ]:
model, X_test, y_test = train(df)
print(f'Test set: {len(X_test):,} rows | positives: {y_test.sum():,}')

---
## 5. Evaluate


In [ ]:
metrics = evaluate(model, X_test, y_test)
print()
print(json.dumps(metrics, indent=2))

---
## 6. ROC curve + Precision-Recall curve


In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax1, color='steelblue')
ax1.plot([0,1],[0,1],'--',color='grey',linewidth=1)
ax1.set_title(f"ROC Curve  (AUC = {metrics['roc_auc']})", fontsize=13, fontweight='bold')

PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=ax2, color='crimson')
ax2.axvline(metrics[f'threshold_{RETURN_PROB_THRESHOLD}']['recall'],
            color='black', linestyle='--', linewidth=1,
            label=f'threshold={RETURN_PROB_THRESHOLD}')
ax2.set_title(f"Precision-Recall Curve  (PR-AUC = {metrics['pr_auc']})",
              fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)

plt.suptitle('Return classifier evaluation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/roc_pr_curves.png')

---
## 7. Confusion matrix @ threshold 0.30


In [ ]:
y_pred = (y_prob >= RETURN_PROB_THRESHOLD).astype(int)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['No return','Return']).plot(
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title(f'Confusion Matrix @ threshold {RETURN_PROB_THRESHOLD}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/confusion_matrix.png')

---
## 8. Feature importance

Using XGBoost's built-in `feature_importances_` (gain-based).  
The calibrator wraps the base estimator — we access it via `.calibrated_classifiers_`.


In [ ]:
feature_names = CAT_COLS + NUM_COLS

# Average importances across the 3 CV folds in CalibratedClassifierCV
importances = np.mean(
    [clf.estimator.feature_importances_
     for clf in model.calibrated_classifiers_],
    axis=0,
)
imp_df = (
    pd.DataFrame({'feature': feature_names, 'importance': importances})
    .sort_values('importance', ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp_df['feature'], imp_df['importance'],
        color='steelblue', edgecolor='white')
ax.set_xlabel('Feature importance (gain)', fontsize=12)
ax.set_title('XGBoost feature importance — return classifier',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/feature_importance.png')
print()
print(imp_df.sort_values('importance', ascending=False).to_string(index=False))


---
## 9. Return probability distribution


In [ ]:
return_probs = model.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(return_probs[y_test == 0], bins=50, alpha=0.6,
        color='steelblue', label='No return (actual)')
ax.hist(return_probs[y_test == 1], bins=50, alpha=0.6,
        color='crimson', label='Return (actual)')
ax.axvline(RETURN_PROB_THRESHOLD, color='black', linestyle='--',
           linewidth=1.5, label=f'Threshold = {RETURN_PROB_THRESHOLD}')
ax.set_xlabel('Predicted return probability', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Return probability distribution by actual label',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/return_prob_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/return_prob_distribution.png')

---
## 10. Add return_prob to master_df → save as master_df_v3

`return_flag = 1` where `return_prob >= 0.30`.  
These orders are pre-assigned pickup slots in the SDVRP (Vybhav → Pritam Day 4 handoff).


In [ ]:
df_v3 = add_return_prob(model, df)

print(f'\nreturn_prob stats:')
print(df_v3['return_prob'].describe().round(4))
print(f'\nreturn_flag distribution:')
print(df_v3['return_flag'].value_counts())

---
## 11. Save all outputs


In [ ]:
# master_df_v3
df_v3.to_parquet(f'{DATA_DIR}/master_df_v3.parquet', index=False)
print(f'data/master_df_v3.parquet  ({len(df_v3):,} rows, {df_v3.shape[1]} cols)')

# model
save_model(model, path=f'{OUTPUT_DIR}/return_clf_v1.pkl')

# metrics
save_metrics(metrics, path=f'{OUTPUT_DIR}/return_classifier_metrics.json')

---
## Deliverables checklist

| Output | Status |
|--------|--------|
| `data/master_df_v3.parquet` (with `return_prob`, `return_flag`) | ✅ |
| `outputs/return_clf_v1.pkl` | ✅ |
| `outputs/return_classifier_metrics.json` | ✅ |
| `outputs/roc_pr_curves.png` | ✅ |
| `outputs/confusion_matrix.png` | ✅ |
| `outputs/feature_importance.png` | ✅ |
| `outputs/return_prob_distribution.png` | ✅ |

**Handoffs:**
- `master_df_v3.parquet` → **Pritam** (SDVRP: `return_flag=1` orders get pickup slots)
- `feature_importance.png` → **Varsha** (report slide)
- `return_classifier_metrics.json` → **Pranav** (RESULTS.md)
